In [2]:
import numpy as np
import cv2
from scipy.spatial.transform import Rotation as R

def compute_camera_pose(image_points, robot_points, camera_matrix, dist_coeffs=None):
    """
    计算相机到机器人坐标系的变换矩阵
    
    参数：
        image_points: 图像上的2D点坐标 (Nx2) [像素]
        robot_points: 机器人坐标系下的3D点坐标 (Nx3) [米]
        camera_matrix: 相机内参矩阵 (3x3)
        dist_coeffs: 畸变系数 (默认None)
    
    返回：
        T_robot_cam: 4x4变换矩阵 (机器人坐标系->相机坐标系)
        T_cam_robot: 4x4变换矩阵 (相机坐标系->机器人坐标系)
        error: 重投影误差
    """
    # 参数检查
    assert len(image_points) >= 4, "至少需要4个点"
    assert len(image_points) == len(robot_points), "点数量不匹配"
    
    # 转换为OpenCV需要的格式
    image_pts = np.array(image_points, dtype=np.float32).reshape(-1,1,2)
    robot_pts = np.array(robot_points, dtype=np.float32).reshape(-1,3)
    print(image_pts)
    print(robot_pts)
    
    # 使用PnP求解（推荐使用SOLVEPNP_ITERATIVE或SOLVEPNP_EPNP）
    success, rvec, tvec = cv2.solvePnP(
        robot_pts, 
        image_pts, 
        camera_matrix, 
        dist_coeffs,
        flags=cv2.SOLVEPNP_ITERATIVE
    )
    
    if not success:
        raise ValueError("PnP求解失败")
    
    # 计算重投影误差
    reprojected_pts, _ = cv2.projectPoints(robot_pts, rvec, tvec, camera_matrix, dist_coeffs)
    error = np.mean(np.linalg.norm(image_pts - reprojected_pts, axis=2))
    
    # 构建变换矩阵（相机坐标系->机器人坐标系）
    R_cam_robot, _ = cv2.Rodrigues(rvec)
    T_cam_robot = np.eye(4)
    T_cam_robot[:3, :3] = R_cam_robot
    T_cam_robot[:3, 3] = tvec.flatten()
    
    # 计算逆变换（机器人坐标系->相机坐标系）
    T_robot_cam = np.linalg.inv(T_cam_robot)
    
    return T_robot_cam, T_cam_robot, error

def project_points_to_image(robot_points, T_cam_robot, camera_matrix, dist_coeffs):
    """
    将机器人坐标系下的3D点投影到图像平面上。
    
    参数:
    - robot_points: 在机器人坐标系中的3D点列表，形状为 (N, 3)。
    - T_cam_robot: 相机到机器人坐标系的变换矩阵 (4x4)。
    - camera_matrix: 相机内参矩阵。
    - dist_coeffs: 相机畸变系数。
    
    返回:
    - image_points: 投影到图像平面上的2D点列表，形状为 (N, 2)。
    """
    # 将机器人坐标系下的点转换为齐次坐标
    robot_points_homogeneous = np.hstack([robot_points, np.ones((len(robot_points), 1))])
    
    # 转换到相机坐标系
    camera_points_homogeneous = T_cam_robot @ robot_points_homogeneous.T
    
    # 转换回非齐次坐标
    camera_points = camera_points_homogeneous[:3, :] / camera_points_homogeneous[3, :]
    
    # 将相机坐标系下的点投影到归一化图像平面
    image_points_normalized = camera_points[:2, :] / camera_points[2, :]
    
    # 使用相机内参矩阵将归一化坐标转换为像素坐标，并考虑畸变
    image_points, _ = cv2.projectPoints(
        camera_points[:3, :].T,  # 需要转置以匹配cv2.projectPoints的输入格式
        np.zeros((3, 1)),        # 无旋转（因为已经是相机坐标系）
        np.zeros((3, 1)),        # 无平移（因为已经是相机坐标系）
        camera_matrix,
        dist_coeffs
    )
    
    return image_points.squeeze()

def auto_calibration():
    camera_matrix = np.array([[606.39587, 0, 322.227], [0, 606.0932, 249.1919], [0, 0, 1]])
    
    # 图像上的2D点（像素坐标）
    image_points = [
        [464.07, 161.67], [505.43, 179.98],  # 左上、右上
        [533, 201], [435, 187],  # 右下、左下
        [461, 220], [488, 246]   # 中间两个点
    ]
    print(np.array([0.1545, -0.3088, 0.24063]))
    pt1 = np.array([0.1545, -0.3088, 0.24063])
    pt2 = np.array([0.203, -0.37, 0.24])
    pt3 = np.array([0.232, -0.43, 0.24])
    pt4 = np.array([0.083, -0.397, 0.24])
    pt5 = np.array([0.1119, -0.4878, 0.24])
    pt6 = np.array([0.14, -0.548, 0.24])
    
    # 机器人坐标系下的3D点（米）
    robot_points = [
        pt1, pt2,   # 对应左上、右上
        pt3, pt4,  # 对应右下、左下
        pt5, pt6   # 中间两个点
    ]
    
    # 计算变换矩阵
    T_robot_cam, T_cam_robot, error = compute_camera_pose(
        image_points, robot_points, camera_matrix
    )
    return T_robot_cam, T_cam_robot


# 示例用法
if __name__ == "__main__":
    camera_matrix = np.array([[606.39587402, 0, 322.22705078],
                               [0, 606.09320068, 249.19195557],
                               [0, 0, 1]])
    # 图像上的2D点（像素坐标）
    image_points = [
        [287,113], [167,102],  # 左上、右上
        [388,96], [200,145],  # 右下、左下
        [389,149], [348,86]   # 中间两个点
    ]
    print(np.array([0.1545, -0.3088, 0.24063])) 
    pt1 = np.array([0.034865176233809, -0.4380707039144145, 0.3108710091094988]) 
    pt2 = np.array([-0.1622011324706111, -0.4083089507433242, 0.3108548563984519])
    pt3 = np.array([0.1983327112363369, -0.3920381421832559, 0.310841478086451])
    pt4 = np.array([-0.1118679032961873, -0.4576856807495194, 0.2600065829717169])
    pt5 = np.array([0.2051686972240063, -0.4263309722824024, 0.2244389286378271])
    pt6 = np.array([0.1305490973483158, -0.4137721300483229, 0.34128953833894])

    # 机器人坐标系下的3D点（米）
    robot_points = [
        pt1, pt2,   # 对应左上、右上
        pt3, pt4,  # 对应右下、左下
        pt5, pt6   # 中间两个点
    ]
    
    # 计算变换矩阵
    T_robot_cam, T_cam_robot, error = compute_camera_pose(
        image_points, robot_points, camera_matrix
    )
    
    print("相机到机器人坐标系的变换矩阵 (T_robot_cam):")
    print(T_robot_cam)
    print("\n外参矩阵，机器人到相机坐标系的变换矩阵 (T_cam_robot):")
    print(T_cam_robot)
    print(f"\n重投影误差: {error:.2f} 像素")
    
    cali_rotation = R.from_matrix(T_cam_robot[:3,:3])
    euler_angles = cali_rotation.as_euler('xyz', degrees=False)  # in radians
    print(euler_angles)
    
    
    
    # 验证数据
    # 物体点：在机器人坐标系中的6个3D点
    robot_points = np.array([
        [0.1545, -0.3088, 0.24063],  # 示例点1
        [0.203, -0.37, 0.24],
        [0.14, -0.548, 0.24]
    ], dtype=np.float32)

    # 相机畸变系数（假设已知）
    dist_coeffs = np.zeros((5, 1))  # 假设没有畸变

    # 计算投影到图像平面上的点
    image_points = project_points_to_image(robot_points, T_cam_robot, camera_matrix, dist_coeffs)
    print(image_points)

[ 0.1545  -0.3088   0.24063]
[[[287. 113.]]

 [[167. 102.]]

 [[388.  96.]]

 [[200. 145.]]

 [[389. 149.]]

 [[348.  86.]]]
[[ 0.03486517 -0.4380707   0.310871  ]
 [-0.16220114 -0.40830895  0.31085485]
 [ 0.19833271 -0.39203814  0.31084147]
 [-0.1118679  -0.45768568  0.26000658]
 [ 0.2051687  -0.42633098  0.22443894]
 [ 0.1305491  -0.41377214  0.34128955]]
相机到机器人坐标系的变换矩阵 (T_robot_cam):
[[ 9.99992341e-01 -3.75865956e-03  1.09084802e-03  8.82112718e-02]
 [-3.55597271e-03 -7.56148583e-01  6.54390308e-01 -1.23837034e+00]
 [-1.63478720e-03 -6.54389175e-01 -7.56156158e-01  9.00782366e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

外参矩阵，机器人到相机坐标系的变换矩阵 (T_cam_robot):
[[ 9.99992341e-01 -3.55597271e-03 -1.63478720e-03 -9.11416199e-02]
 [-3.75865956e-03 -7.56148583e-01 -6.54389175e-01 -3.46598194e-01]
 [ 1.09084802e-03  6.54390308e-01 -7.56156158e-01  1.49141346e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

重投影误差: 0.81 像素
[ 2.42821610e+00 -1.